# NB12 — Statistical Significance (bootstrap CI + McNemar)

Bootstrap 95% CIs for macro-F1 and paired McNemar tests between the **final proposed model** and each
comparator (vanilla, backbones, adversarial, author frozen) on **Ben-Sarc binary**, with Bonferroni
correction. CPU/Mac. Auto-discovers prediction files.


In [1]:
import json, re, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
def find_repo_root():
    for c in [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents):
        if (c/"01_data").exists(): return c.resolve()
    raise RuntimeError("repo root not found.")
ROOT=find_repo_root(); TABLES=ROOT/"04_outputs"/"tables"; PRED=ROOT/"04_outputs"/"predictions"; FIGS=ROOT/"04_outputs"/"figures"
for p in [TABLES,FIGS]: p.mkdir(parents=True,exist_ok=True)
TASK="ben_sarc_binary"
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
def collect(task, split):
    out={}
    for f in sorted(PRED.glob(f"*_{task}_{split}_predictions.csv")):
        name=f.name.replace(f"_{task}_{split}_predictions.csv","")
        try: out[name]=pd.read_csv(f).dropna(subset=["text"]).drop_duplicates("text")
        except Exception as e: print("skip",f.name,e)
    return out
def macro(df): return f1_score(df["gold_label"],df["pred_label"],average="macro",zero_division=0)
print("ROOT:",ROOT)

ROOT: /Users/sefayet/Desktop/Github/Sarcasm_detection


In [2]:
from scipy.stats import chi2 as chi2dist
test=collect(TASK,"test")
# pick the proposed model
PROPOSED=None
for cand in ["09b_fgm_awp","09_FINAL_proposed","08_P1_fullft_fgm","07_fgm_awp","07_fgm_e05"]:
    if cand in test: PROPOSED=cand; break
print("proposed =", PROPOSED, "| available:", list(test))
assert PROPOSED, "no proposed-model predictions found (run NB09/08/07 first)"

def boot_ci(y,pred,B=1000,seed=42):
    rng=np.random.default_rng(seed); n=len(y); s=np.empty(B)
    for i in range(B):
        idx=rng.integers(0,n,n); s[i]=f1_score(y[idx],pred[idx],average="macro",zero_division=0)
    return float(s.mean()), float(np.percentile(s,2.5)), float(np.percentile(s,97.5))
def mcnemar(ca,cb):
    b=int(((ca==1)&(cb==0)).sum()); c=int(((ca==0)&(cb==1)).sum()); n=b+c
    if n==0: return 0.0,1.0,b,c
    chi2=(abs(b-c)-1)**2/n; return float(chi2),float(1-chi2dist.cdf(chi2,1)),b,c

proposed = 09b_fgm_awp | available: ['04_bengali-bert', '04_mbert', '04_sagorsarker-bb', '05_baseline_banglabert', '06_banglabert', '06_mbert', '06_muril', '06_sagorsarker-bb', '06_xlm-roberta', '07_awp', '07_fgm_awp', '07_fgm_e05', '07_freelb_k3', '08_P0_fullft', '08_P1_fullft_fgm', '08_P2_fullft_llrd', '08_P3_fullft_fgm_llrd', '08_P4_freeze6_fgm', '08_P5_freeze6_fgm_llrd', '08_P6_maxlen192_fgm_llrd', '08_P7_fgm_llrd_ls', '08_P8_large_fgm_llrd', '08_ZIHAN_repro', '09_FINAL_proposed', '09b_fgm_awp']


In [3]:
pi=test[PROPOSED].set_index("text")
rows=[]; comparators=[k for k in test if k!=PROPOSED]
for k in comparators:
    ci=test[k].set_index("text")
    idx=sorted(set(pi.index)&set(ci.index))
    if len(idx)<50: continue
    y=pi.loc[idx,"gold_label"].values
    pa=pi.loc[idx,"pred_label"].values; pb=ci.loc[idx,"pred_label"].values
    ca=(pi.loc[idx,"correct"].values if "correct" in pi.columns else (pa==y).astype(int))
    cb=(ci.loc[idx,"correct"].values if "correct" in ci.columns else (pb==y).astype(int))
    f_p=f1_score(y,pa,average="macro"); f_c=f1_score(y,pb,average="macro")
    chi2,p,b,c=mcnemar(ca,cb)
    fam=("Lora et al. (reproduced)" if k.startswith(("02_","03_","04_")) else "This work")
    rows.append({"comparator":k,"family":fam,"f1_proposed":f_p,"f1_comparator":f_c,"delta":f_p-f_c,
                 "mcnemar_chi2":chi2,"p_value":p,"n_pairs":len(idx)})
res=pd.DataFrame(rows).sort_values(["family","delta"],ascending=[True,False])
m=len(res); res["p_bonferroni"]=(res["p_value"]*m).clip(upper=1.0)
res["sig_0.05"]=res["p_bonferroni"]<0.05
# bootstrap CI for proposed
y=pi["gold_label"].values; mean,lo,hi=boot_ci(y,pi["pred_label"].values)
res.to_csv(TABLES/"12_significance.csv",index=False)
print("="*90); print(f"  SIGNIFICANCE — proposed = {PROPOSED}"); print("="*90)
print(f"proposed macro-F1 bootstrap mean={mean:.4f}  95% CI=[{lo:.4f},{hi:.4f}]\n")
print(res.to_string(index=False,float_format="%.4f"))
json.dump({"proposed":PROPOSED,"boot_mean":mean,"ci95":[lo,hi]},open(TABLES/"12_proposed_ci.json","w"),indent=2)

  SIGNIFICANCE — proposed = 09b_fgm_awp
proposed macro-F1 bootstrap mean=0.8042  95% CI=[0.7889,0.8192]

              comparator                   family  f1_proposed  f1_comparator  delta  mcnemar_chi2  p_value  n_pairs  p_bonferroni  sig_0.05
                04_mbert Lora et al. (reproduced)       0.8038         0.6916 0.1122      122.5729   0.0000     2563        0.0000      True
       04_sagorsarker-bb Lora et al. (reproduced)       0.8038         0.6921 0.1117      114.7246   0.0000     2563        0.0000      True
         04_bengali-bert Lora et al. (reproduced)       0.8038         0.7444 0.0594       50.3355   0.0000     2563        0.0000      True
                06_mbert                This work       0.8038         0.7428 0.0611       51.6688   0.0000     2563        0.0000      True
       06_sagorsarker-bb                This work       0.8038         0.7510 0.0528       35.3198   0.0000     2563        0.0000      True
          06_xlm-roberta                This work

In [4]:
# Ensemble vs baseline significance (stacking ensemble has no logits; rebuild preds from pooled probs)
from scipy.stats import chi2 as chi2dist
val=collect(TASK,"val")
DROP={"07_fgm_awp","09_FINAL_proposed"}
mods=[k for k,v in test.items() if "prob_1" in v.columns and k not in DROP]
def aligned(coll,keys,col):
    s={k:coll[k].set_index("text")[col] for k in keys}
    idx=sorted(set.intersection(*[set(v.index) for v in s.values()]))
    return pd.DataFrame({k:s[k].loc[idx] for k in keys}),idx
from sklearn.linear_model import LogisticRegression
vmods=[k for k in mods if k in val and "prob_1" in val[k].columns]
Pv,vi=aligned(val,vmods,"prob_1"); yv=val[vmods[0]].set_index("text")["gold_label"].loc[vi].values
Pt,ti=aligned(test,vmods,"prob_1"); yt=test[vmods[0]].set_index("text")["gold_label"].loc[ti].values
lr=LogisticRegression(max_iter=1000).fit(Pv.values,yv); ens_pred=lr.predict(Pt.values)
base=test["05_baseline_banglabert"].set_index("text").loc[ti]
cb=(base["pred_label"].values==yt).astype(int); ce=(ens_pred==yt).astype(int)
def mcnemar(ca,cb):
    b=int(((ca==1)&(cb==0)).sum()); c=int(((ca==0)&(cb==1)).sum()); n=b+c
    if n==0: return 0.0,1.0
    chi2=(abs(b-c)-1)**2/n; return float(chi2),float(1-chi2dist.cdf(chi2,1))
chi2,p=mcnemar(ce,cb)
print(f"ENSEMBLE(stacking) vs baseline: F1 {f1_score(yt,ens_pred,average='macro'):.4f} vs {f1_score(yt,base['pred_label'],average='macro'):.4f} | chi2={chi2:.3f} p={p:.4f}")
pd.DataFrame([{"comparison":"ensemble_vs_baseline","mcnemar_chi2":chi2,"p_value":p}]).to_csv(TABLES/"12_ensemble_significance.csv",index=False)

ENSEMBLE(stacking) vs baseline: F1 0.8115 vs 0.8025 | chi2=2.564 p=0.1093
